# 隔离模拟的可信离线分发与启动

此入口由实验管理员执行，负责将公开单表分发为两方输入，不属于联邦运行中的数据参与方。随后各方在系统沙箱执行自己的 Notebook。中央只读取固定格式回执，不收集私有清洗表或参数。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json, sys, yaml
repo = Path.cwd()
while not (repo / "modules/m1_data_selection").is_dir():
    if repo == repo.parent:
        raise RuntimeError("请从项目仓库运行")
    repo = repo.parent
sys.path.insert(0, str(repo))
from modules.m1_data_selection.components.isolated_simulation import run_isolated
configs = [repo / "modules/m1_data_selection/configs" / name for name in ("uci_bank_preparation.yaml", "hillstrom_preparation.yaml")]
run_dir = repo.parent / "8-9月/实验日志" / ("launcher_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex)
run_dir.mkdir(parents=True, exist_ok=False)
print("VFL_RUN_LOCATOR=" + json.dumps({"result_dir": str(run_dir)}))
print("config:", [str(p) for p in configs])
print("seeds:", [yaml.safe_load(p.read_text())["seeds"] for p in configs])
print("git: each party config records git_sha and exact code_sha256")

此启动器不执行模型训练。完整原件只由可信的公开数据分发阶段访问，各方处理进程的输入权限由操作系统限制。

In [ ]:
journals = []
for config_path in configs:
    journal = run_isolated(config_path, repo)
    journals.append(str(journal))
    print("完成数据准备：", journal)
(run_dir / "experiments.json").write_text(json.dumps(journals, ensure_ascii=False, indent=2))

每个实验都有独立登记和两方回执。任一隔离测试失败时抛出错误并写 FAILED，不允许退回集中式处理。

In [ ]:
for journal in journals:
    print(json.dumps(json.loads((Path(journal) / "summary.json").read_text()), ensure_ascii=False, indent=2))

data_preparation_only 只证明本次数据准备完成。联合训练、两主机物理隔离和协议安全仍须单独验收。